In [2]:
# Install the required Mamba kernels
!pip install mamba-ssm causal-conv1d>=1.2.0 --no-cache-dir

^C


In [1]:
# cell1 — Setup, data loading, and model initialization
import os
import gc
import json
import math
import torch
import mamba_ssm
import kagglehub
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# ─── Hyperparameters ───
LORA_RANK = 32
MAX_SEQ_LEN = 4096
NUM_EPOCHS = 3
GRAD_ACCUM = 8
LR = 2e-4
BATCH_SIZE = 1
WARMUP_RATIO = 0.05

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Load SFT data (pre-generated with CoT reasoning) ───
SFT_TRAIN_PATH = "/kaggle/input/sft-training-data/sft_train.jsonl"
SFT_VAL_PATH = "/kaggle/input/sft-training-data/sft_val.jsonl"

# Fallback: load from train.csv if SFT data not available
if not os.path.exists(SFT_TRAIN_PATH):
    print("SFT data not found, falling back to train.csv with simple formatting")
    USE_RAW_CSV = True
else:
    USE_RAW_CSV = False

# cell2 — Triton/ptxas-blackwell workarounds (MUST KEEP)
import shutil, stat, sys

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
shutil.copy2(src, dst)
os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

import triton.backends.nvidia as nv_backend
src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
dst_bin = "/tmp/triton_nvidia_bin"
shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
for f in os.listdir(dst_bin):
    fp = os.path.join(dst_bin, f)
    if os.path.isfile(fp):
        os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
os.environ["TRITON_PTXAS_PATH"] = dst

# ─── Load model ───
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Force slow path — bypass broken CUDA kernels (MUST KEEP)
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# ─── LoRA config ───
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=64,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable gradient checkpointing with use_reentrant=False (MUST KEEP)
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# cell3 — Data preparation

SYSTEM_PROMPT = (
    "You are a systematic reasoning assistant. For each puzzle, carefully "
    "analyze the examples to discover the underlying rule, show your reasoning "
    "step by step inside <think>...</think> tags, and always place your final "
    "answer inside \\boxed{}. Do not include \\boxed{} anywhere else in your response."
)


def build_text_from_messages(messages):
    """Build training text using the tokenizer's chat template."""
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        parts = []
        for msg in messages:
            parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
        text = "\n".join(parts)
    return text


def load_sft_data(path):
    """Load SFT data from JSONL and convert to training texts."""
    texts = []
    with open(path, "r") as f:
        for line in f:
            ex = json.loads(line)
            text = build_text_from_messages(ex["messages"])
            texts.append(text)
    return texts


def load_raw_csv_data():
    """Fallback: load from train.csv with CoT template."""
    import polars as pl
    train = pl.read_csv(
        "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
    )
    texts = []
    for row in train.iter_rows(named=True):
        prompt = row["prompt"]
        answer = row["answer"]
        assistant_msg = (
            f"<think>\nAnalyzing the puzzle examples to find the pattern.\n"
            f"After careful analysis, I can determine the answer.\n</think>\n\n"
            f"The answer is \\boxed{{{answer}}}"
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": assistant_msg},
        ]
        texts.append(build_text_from_messages(messages))
    return texts


if USE_RAW_CSV:
    train_texts = load_raw_csv_data()
    val_texts = train_texts[-500:]
    train_texts = train_texts[:-500]
else:
    train_texts = load_sft_data(SFT_TRAIN_PATH)
    val_texts = load_sft_data(SFT_VAL_PATH)

print(f"Training examples: {len(train_texts)}")
print(f"Validation examples: {len(val_texts)}")
print(f"Example text (first 500 chars):\n{train_texts[0][:500]}")


class SFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = []
        skipped = 0
        for text in texts:
            enc = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )
            ids = enc["input_ids"].squeeze(0)
            mask = enc["attention_mask"].squeeze(0)

            if ids.sum() == 0:
                skipped += 1
                continue

            labels = ids.clone()
            labels[mask == 0] = -100

            self.encodings.append({
                "input_ids": ids,
                "attention_mask": mask,
                "labels": labels,
            })

        print(f"Tokenized {len(self.encodings)} examples (skipped {skipped})")

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


train_dataset = SFTDataset(train_texts, tokenizer, MAX_SEQ_LEN)
val_dataset = SFTDataset(val_texts, tokenizer, MAX_SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

del train_texts, val_texts
gc.collect()

# cell4 — Training loop with cosine scheduler and validation
import torch.nn.functional as F

# Bypass Triton rmsnorm (MUST KEEP)
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

model.train()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps = (len(train_loader) * NUM_EPOCHS) // GRAD_ACCUM
warmup_steps = int(total_steps * WARMUP_RATIO)


def cosine_lr(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=cosine_lr)

print(
    f"Training: {NUM_EPOCHS} epochs, ~{total_steps} optimizer steps, "
    f"{len(train_dataset)} examples, warmup={warmup_steps} steps"
)

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for i, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        running_loss += outputs.loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            step = (i + 1) // GRAD_ACCUM
            if step % 50 == 0:
                avg = running_loss / (i + 1)
                lr_now = scheduler.get_last_lr()[0]
                print(
                    f"  epoch {epoch+1} | step {step}/{total_steps} | "
                    f"avg_loss {avg:.4f} | lr {lr_now:.2e}"
                )

    if (i + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss_total += outputs.loss.item()
    avg_val_loss = val_loss_total / max(1, len(val_loader))

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} — "
        f"train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(OUTPUT_DIR)
        print(f"  Saved best model (val_loss={best_val_loss:.4f})")

    gc.collect()
    torch.cuda.empty_cache()

# cell5 — Package submission
import zipfile
import json as _json

model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# Verify adapter_config.json constraints
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path, "r") as cf:
    config = _json.load(cf)
assert config.get("r", 0) <= 32, f"LoRA rank {config.get('r')} > 32 — will be rejected!"
print(f"LoRA rank verified: r={config.get('r')}")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    assert "adapter_model.safetensors" in names, "Missing adapter_model.safetensors!"
    print("submission.zip verified OK")

# cell6 — GRPO Stage (optional, run after SFT if time allows)
# Uncomment the following to add GRPO training after SFT

"""
import re
from trl import GRPOTrainer, GRPOConfig

def extract_boxed_answer(text: str) -> str | None:
    matches = re.findall(r'\\\\boxed\\{([^}]*)\\}', text)
    return matches[-1].strip() if matches else None

def reward_correctness(completions, ground_truths, **kwargs) -> list[float]:
    rewards = []
    for completion, gt in zip(completions, ground_truths):
        predicted = extract_boxed_answer(completion)
        if predicted is None:
            rewards.append(0.0)
        elif predicted.strip() == gt.strip():
            rewards.append(1.0)
        else:
            try:
                rel_diff = abs(float(predicted) - float(gt)) / (abs(float(gt)) + 1e-9)
                rewards.append(1.0 if rel_diff < 0.01 else 0.0)
            except (ValueError, TypeError):
                rewards.append(0.0)
    return rewards

def reward_format(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        score = 0.0
        if '<think>' in completion and '</think>' in completion:
            score += 0.3
        if '\\\\boxed{' in completion:
            score += 0.7
        rewards.append(score)
    return rewards

grpo_config = GRPOConfig(
    output_dir="/kaggle/working/grpo_adapter",
    learning_rate=5e-6,
    num_generations=4,  # reduce from 8 if OOM
    temperature=0.7,
    kl_coef=0.01,
    max_new_tokens=1024,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=True,
    logging_steps=10,
    save_steps=500,
)

# Load GRPO dataset (same format but needs 'prompt' and 'answer' columns)
# grpo_trainer = GRPOTrainer(
#     model=model,
#     config=grpo_config,
#     reward_funcs=[reward_correctness, reward_format],
#     train_dataset=grpo_dataset,
# )
# grpo_trainer.train()
"""


ModuleNotFoundError: No module named 'mamba_ssm'